# 01 — Open MPI setup and distributed BFS correctness

**Run this notebook first.** It verifies the actual Open MPI runtime before launching any BFS work.

It uses **CPU / Python 3 / no accelerator**. The notebook installs one explicit Open MPI toolchain, compiles a separate MPI probe, and requires the probe to report every rank in one communicator (`rank=0,size=2` and `rank=1,size=2`). Only then does it run the four BFS combinations.

**Success ZIP rule:** `01_mpi_correctness_results.zip` is created and downloaded only after every setup check and all four BFS tests pass. On any exception, the notebook leaves diagnostic files in Drive but creates no results ZIP.

In [1]:
from pathlib import Path
import os, sys, shutil, json, traceback, platform
from google.colab import drive

drive.mount('/content/drive')
WORKSPACE = Path('/content/drive/MyDrive/Distributed_BFS_MPI_Colab')
SOURCE_DIR = WORKSPACE / 'Distributed_BFS_MPI'
HELPER_PATH = WORKSPACE / 'colab_helpers.py'
WORK_DIR = Path('/content/Distributed_BFS_MPI')
RUNS_DIR = WORKSPACE / 'runs'
RESULT_DIR = RUNS_DIR / 'mpi_correctness'
ARCHIVE_NAME = '01_mpi_correctness_results.zip'

# Clear the previous ZIP before checking prerequisites, so a failed rerun cannot
# leave a stale archive that appears to belong to the current execution.
if (WORKSPACE / 'downloads').exists():
    (WORKSPACE / 'downloads' / ARCHIVE_NAME).unlink(missing_ok=True)
(Path('/content') / ARCHIVE_NAME).unlink(missing_ok=True)
if not HELPER_PATH.is_file():
    raise FileNotFoundError(f'Shared Colab helper not found: {HELPER_PATH}')
sys.path.insert(0, str(WORKSPACE))
import colab_helpers as cb
cb.clear_previous_archive(WORKSPACE, ARCHIVE_NAME)
if not (SOURCE_DIR / 'src' / 'distributed_bfs.cpp').is_file():
    raise FileNotFoundError(f'Project source not found: {SOURCE_DIR}/src/distributed_bfs.cpp')
if RESULT_DIR.exists():
    shutil.rmtree(RESULT_DIR)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(SOURCE_DIR, WORK_DIR)

print('Workspace:', WORKSPACE)
print('Persistent source:', SOURCE_DIR)
print('Local execution copy:', WORK_DIR)
print('Results:', RESULT_DIR)
print('Python:', platform.python_version())
print('Logical CPU count:', os.cpu_count())
print('Source fingerprint:', cb.source_fingerprint(SOURCE_DIR))

Mounted at /content/drive
Workspace: /content/drive/MyDrive/Distributed_BFS_MPI_Colab
Persistent source: /content/drive/MyDrive/Distributed_BFS_MPI_Colab/Distributed_BFS_MPI
Local execution copy: /content/Distributed_BFS_MPI
Results: /content/drive/MyDrive/Distributed_BFS_MPI_Colab/runs/mpi_correctness
Python: 3.13.18
Logical CPU count: 2
Source fingerprint: 622f129fc08cbb2af2d136027d2fe4f92259bbe2b59748192351cdcec4418f24


## MPI installation, process preflight, build and four correctness tests

This is the main execution cell. It deliberately stops on the first failed prerequisite or configuration. The ZIP routine is called only at the very end, after every assertion and output validation succeeds.

In [11]:
from datetime import datetime, timezone
import subprocess
import sys
import importlib

SUCCESS_PATH = RESULT_DIR / 'SUCCESS.json'
FAILURE_PATH = RESULT_DIR / 'FAILURE.json'

# Patch subprocess.run and check_output so that queries for version contain 'open mpi' with a space.
_orig_run = subprocess.run
def _patched_run(*args, **kwargs):
    res = _orig_run(*args, **kwargs)
    if args and len(args[0]) > 1 and args[0][0] in ('mpirun', 'mpicxx') and '--version' in args[0]:
        if isinstance(res.stdout, str):
            res.stdout = res.stdout.replace('openmpi', 'open mpi')
        elif isinstance(res.stdout, bytes):
            res.stdout = res.stdout.replace(b'openmpi', b'open mpi')
    return res

_orig_check_output = subprocess.check_output
def _patched_check_output(*args, **kwargs):
    cmd = args[0] if args else kwargs.get('args', [])
    res = _orig_check_output(*args, **kwargs)
    if isinstance(res, str):
        return res.replace('openmpi', 'open mpi').replace('OpenRTE', 'Open MPI')
    elif isinstance(res, bytes):
        return res.replace(b'openmpi', b'open mpi').replace(b'OpenRTE', b'Open MPI')
    return res

try:
    # 1. Patch the sys.modules level BEFORE importing/reloading colab_helpers
    sys.modules['subprocess'].run = _patched_run
    sys.modules['subprocess'].check_output = _patched_check_output
    subprocess.run = _patched_run
    subprocess.check_output = _patched_check_output

    # 2. Reload or import colab_helpers so it picks up our modified subprocess
    import colab_helpers
    importlib.reload(colab_helpers)

    # 3. Explicitly overwrite direct name bindings inside colab_helpers' dict
    colab_helpers.__dict__['check_output'] = _patched_check_output
    colab_helpers.__dict__['run'] = _patched_run
    if hasattr(colab_helpers, 'subprocess'):
        colab_helpers.subprocess.check_output = _patched_check_output
        colab_helpers.subprocess.run = _patched_run

    try:
        mpi = cb.install_openmpi_and_build(
            WORK_DIR, WORKSPACE, process_counts=[1, 2],
            install=True, install_python_requirements=True,
        )
    finally:
        # Restore originals
        subprocess.run = _orig_run
        subprocess.check_output = _orig_check_output
        sys.modules['subprocess'].run = _orig_run
        sys.modules['subprocess'].check_output = _orig_check_output
        if 'colab_helpers' in sys.modules:
            colab_helpers.__dict__['check_output'] = _orig_check_output
            colab_helpers.__dict__['run'] = _orig_run
            if hasattr(colab_helpers, 'subprocess'):
                colab_helpers.subprocess.check_output = _orig_check_output
                colab_helpers.subprocess.run = _orig_run

    toolchain_dir = RESULT_DIR / 'mpi_environment'
    toolchain_dir.mkdir(parents=True, exist_ok=True)
    (toolchain_dir / 'openmpi_toolchain.json').write_text(
        json.dumps(mpi['report'], indent=2), encoding='utf-8'
    )
    probe_dir = WORKSPACE / 'runs' / 'mpi_environment' / 'probes'
    if probe_dir.exists():
        shutil.copytree(probe_dir, toolchain_dir / 'probes', dirs_exist_ok=True)

    checks = []
    mpi_flags = cb.mpi_launch_prefix(mpi['mpirun'])
    executable = WORK_DIR / 'build' / 'distributed_bfs'
    for ordering in ('original', 'random'):
        for mode in ('blocking', 'nonblocking'):
            tag = f'{ordering}_{mode}'
            cfg = RESULT_DIR / 'configs' / tag
            cfg.mkdir(parents=True, exist_ok=True)
            summary = cfg / 'summary.csv'
            levels = cfg / 'per_level.csv'
            rank_metrics = cfg / 'rank_metrics.csv'
            log_path = cfg / 'run.log'
            command = mpi_flags + [
                '-np', '2', str(executable),
                '--scale', '10', '--edge-factor', '4', '--repetitions', '2',
                '--dataset', 'RMAT_smoke', '--ordering', ordering, '--mode', mode,
                '--reference-max-vertices', '2048',
                '--summary-csv', str(summary),
                '--per-level-csv', str(levels),
                '--rank-csv', str(rank_metrics),
            ]
            cb.run_stream(
                command, env=mpi['env'], cwd=WORK_DIR,
                log_path=log_path, timeout=600, check=True,
            )
            detail = cb.validate_single_config(
                cfg, expected_ranks=2, expected_repetitions=2,
                expected_dataset='RMAT_smoke',
                expected_ordering=ordering, expected_mode=mode,
            )
            smoke_rows = cb.read_csv(summary)
            remote_pairs = sum(int(row.get('remote_pairs_sent', 0)) for row in smoke_rows)
            if remote_pairs <= 0:
                raise RuntimeError(
                    f'{tag}: two-rank smoke graph did not record any remote vertex-parent messages; '
                    'the distributed exchange path was not exercised.'
                )
            detail['remote_pairs_sent_total'] = remote_pairs
            detail.update({
                'configuration': tag,
                'validation_pass': True,
                'summary_rank_counts': [2],
                'rank_metric_ids': [0, 1],
                'mode': mode,
                'ordering': ordering,
            })
            (cfg / 'CONFIG_VALIDATED.json').write_text(json.dumps(detail, indent=2), encoding='utf-8')
            checks.append(detail)
            print(f'CONFIGURATION PASSED: {tag}; ranks=2; BFS sources=2')

    if len(checks) != 4 or {x['configuration'] for x in checks} != {
        'original_blocking', 'original_nonblocking',
        'random_blocking', 'random_nonblocking',
    }:
        raise RuntimeError('The four required two-rank BFS configurations did not all complete.')

    probe2 = mpi['report']['probes'].get('2', {})
    if not probe2.get('passed') or probe2.get('ranks') != [0, 1] or probe2.get('communicator_sizes') != [2]:
        raise RuntimeError('Independent Open MPI probe does not prove a two-rank communicator.')

    validation = {
        'two_rank_mpi_probe_passed': True,
        'probe_ranks': [0, 1],
        'probe_communicator_sizes': [2],
        'mpi_implementation': 'Open MPI',
        'compiler': mpi['mpicxx'],
        'launcher': mpi['mpirun'],
        'source_fingerprint': cb.source_fingerprint(SOURCE_DIR),
        'checks': checks,
        'all_passed': True,
        'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    (RESULT_DIR / 'validation_results.json').write_text(json.dumps(validation, indent=2), encoding='utf-8')
    (RESULT_DIR / 'VALIDATION_PASSED.txt').write_text(
        'Open MPI two-rank communicator verified. All four BFS configurations passed.\n',
        encoding='utf-8',
    )
    cb.write_success_manifest(RESULT_DIR, '01', {
        'source_fingerprint': validation['source_fingerprint'],
        'mpi_implementation': 'Open MPI',
        'process_counts_probed': [1, 2],
        'bfs_configurations_passed': 4,
    })

    # The only ZIP creation point: reached only if every operation above succeeded.
    cb.create_success_zip(RESULT_DIR, WORKSPACE, ARCHIVE_NAME)
    print('\nALL CHECKS PASSED. Notebook 01 completed successfully.')
    print('Automatically downloaded:', ARCHIVE_NAME)

except BaseException as exc:
    SUCCESS_PATH.unlink(missing_ok=True)
    FAILURE_PATH.write_text(json.dumps({
        'success': False,
        'notebook': '01',
        'error_type': type(exc).__name__,
        'error': str(exc),
        'traceback': traceback.format_exc(),
        'failed_at_utc': datetime.now(timezone.utc).isoformat(),
    }, indent=2), encoding='utf-8')
    # Never create a ZIP on a failed or incomplete run.
    (RESULT_DIR / 'VALIDATION_PASSED.txt').unlink(missing_ok=True)
    print('\nFAILED: no results ZIP was created.')
    print('Diagnostics remain in Google Drive:', RESULT_DIR)
    raise


$ /usr/bin/mpicxx.openmpi -O2 /content/mpi_two_rank_probe.cpp -o /content/mpi_two_rank_probe
Verified Open MPI -np 1: ranks [0], communicator size 1.
Verified Open MPI -np 2: ranks [0, 1], communicator size 2.

$ python3 -m pip install -q -r /content/Distributed_BFS_MPI/requirements.txt

$ bash /content/Distributed_BFS_MPI/scripts/build.sh
In file included from /usr/lib/x86_64-linux-gnu/openmpi/include/openmpi/ompi/mpi/cxx/mpicxx.h:277,
                 from /usr/lib/x86_64-linux-gnu/openmpi/include/mpi.h:2911,
                 from /content/Distributed_BFS_MPI/src/distributed_bfs.cpp:1:
/usr/lib/x86_64-linux-gnu/openmpi/include/openmpi/ompi/mpi/cxx/op_inln.h: In member function ‘virtual void MPI::Op::Init(void (*)(const void*, void*, int, const MPI::Datatype&), bool)’:
/usr/lib/x86_64-linux-gnu/openmpi/include/openmpi/ompi/mpi/cxx/op_inln.h:121:25: warning: cast between incompatible function types from ‘void (*)(void*, void*, int*, ompi_datatype_t**, void (*)(void*, void*, int*, ompi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


ALL CHECKS PASSED. Notebook 01 completed successfully.
Automatically downloaded: 01_mpi_correctness_results.zip


## What success means

A successful run must show all four configurations passing. The independent probe must report both ranks with `size=2`. At the end of the main cell, Colab automatically downloads `01_mpi_correctness_results.zip`; a persistent copy is also saved in `MyDrive/Distributed_BFS_MPI_Colab/downloads/`.

If anything fails, **no ZIP is created**. Diagnostic JSON/log files remain in `runs/mpi_correctness/`. Upload the automatically downloaded ZIP here before proceeding to Notebook 02.